# Week 3 Improvement: Boost CAMELYON16 Baseline AUC

**Target:** Improve from AUC=0.699 → 0.75+

**Approach:**
1. Hyperparameter tuning (lr, batch size, epochs, hidden dims)
2. Deeper GNN architecture (3 layers, regularization)
3. Different graph construction (k-NN variants)
4. Ensemble predictions if needed


In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt, json
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score
import warnings
import sys
from scipy.spatial import KDTree

warnings.filterwarnings('ignore')
torch.manual_seed(42); np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Setup paths
FEATURES_DIR = Path('./data/features')
OUT_DIR = Path('./outputs')
CKPT_DIR = Path('./checkpoints')
OUT_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

# Find project root
current = Path.cwd()
for candidate in [current, current.parent, current.parent.parent]:
    if (candidate / 'pathq').exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GCNConv
from sklearn.model_selection import train_test_split

print(f'Setup complete. Features: {len(list(FEATURES_DIR.glob("*_features.pt")))}')

Device: cuda
Setup complete. Features: 333


## 1. Load Dataset (Same as Week 3)

In [2]:
class FeatureGraphDataset(torch.utils.data.Dataset):
    """Load pre-extracted features and build K-NN graphs."""
    def __init__(self, feature_files, k=8):
        self.feature_files = sorted(feature_files)
        self.k = k
        self.graphs = []
        
        for feat_file in self.feature_files:
            stem = feat_file.stem
            label = 1 if 'tumor' in stem else 0
            
            if 'test' in stem.lower():
                continue
            
            data = torch.load(feat_file, weights_only=False)
            features = data['features']
            coords = data['coords']
            
            # K-NN with bidirectional edges
            tree = KDTree(coords.numpy())
            _, indices = tree.query(coords.numpy(), k=self.k+1)
            indices = indices[:, 1:]
            
            edge_list = []
            for node_idx, neighbors in enumerate(indices):
                for neighbor_idx in neighbors:
                    edge_list.append([node_idx, neighbor_idx])
                    edge_list.append([neighbor_idx, node_idx])
            
            edge_set = set(map(tuple, edge_list))
            edges = torch.tensor(list(edge_set), dtype=torch.long).t().contiguous()
            
            graph = Data(
                x=features,
                edge_index=edges,
                y=torch.tensor(label, dtype=torch.long),
                coords=coords,
                slide_id=stem
            )
            self.graphs.append(graph)
    
    def __len__(self):
        return len(self.graphs)
    
    def __getitem__(self, idx):
        return self.graphs[idx]

# Load and split
feature_files = sorted(FEATURES_DIR.glob('*_features.pt'))
all_graphs = FeatureGraphDataset(feature_files, k=8)

all_labels = [g.y.item() for g in all_graphs.graphs]
indices = list(range(len(all_graphs)))

# Stratified split
train_idx, temp_idx, _, _ = train_test_split(
    indices, all_labels, test_size=0.30, stratify=all_labels, random_state=42
)

temp_labels = [all_labels[i] for i in temp_idx]
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, temp_labels, test_size=0.50, stratify=temp_labels, random_state=42
)

train_ds = [all_graphs.graphs[i] for i in train_idx]
val_ds = [all_graphs.graphs[i] for i in val_idx]
test_ds = [all_graphs.graphs[i] for i in test_idx]

print(f'Splits: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')
print(f'Train labels: {sum(1 for g in train_ds if g.y.item()==1)}/{len(train_ds)} tumor')
print(f'Val labels:   {sum(1 for g in val_ds if g.y.item()==1)}/{len(val_ds)} tumor')
print(f'Test labels:  {sum(1 for g in test_ds if g.y.item()==1)}/{len(test_ds)} tumor')

Splits: train=154, val=33, test=34
Train labels: 77/154 tumor
Val labels:   17/33 tumor
Test labels:  17/34 tumor


## 2. Define Model Variants

In [ ]:
class ABMILAttention(nn.Module):
    """Gated attention (Ilse et al. 2018)."""
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.V = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh())
        self.U = nn.Sequential(nn.Linear(dim, hidden), nn.Sigmoid())
        self.w = nn.Linear(hidden, 1, bias=False)

    def forward(self, x, batch_idx):
        A = self.w(self.V(x) * self.U(x))
        B = batch_idx.max().item() + 1
        slide_feats, attn_out = [], torch.zeros_like(A)
        for b in range(B):
            mask = (batch_idx == b)
            w_b  = torch.softmax(A[mask], dim=0)
            attn_out[mask] = w_b
            slide_feats.append((w_b * x[mask]).sum(0, keepdim=True))
        return torch.cat(slide_feats, 0), attn_out


class ClassicalGNN(nn.Module):
    """2-layer baseline."""
    def __init__(self, in_dim=2048, hidden=256, n_classes=2, dropout=0.3):
        super().__init__()
        self.proj  = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(dropout)
        )
        self.conv1 = GCNConv(hidden, hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.drop  = nn.Dropout(dropout)
        self.attn  = ABMILAttention(hidden, hidden // 2)
        self.head  = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, n_classes)
        )

    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        x  = self.proj(x)
        h  = self.drop(F.relu(self.bn1(self.conv1(x, ei)))); x = x + h
        h  = F.relu(self.bn2(self.conv2(x, ei)));            x = x + h
        sf, attn = self.attn(x, batch)
        return self.head(sf), attn


class DeeperGNN(nn.Module):
    """3-layer with stronger regularization."""
    def __init__(self, in_dim=2048, hidden=256, n_classes=2, dropout=0.4):
        super().__init__()
        self.proj  = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(dropout)
        )
        self.conv1 = GCNConv(hidden, hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.conv3 = GCNConv(hidden, hidden)
        self.bn3   = nn.BatchNorm1d(hidden)
        self.drop  = nn.Dropout(dropout)
        self.attn  = ABMILAttention(hidden, hidden // 2)
        self.head  = nn.Sequential(
            nn.Linear(hidden, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )

    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        x  = self.proj(x)
        h  = self.drop(F.relu(self.bn1(self.conv1(x, ei)))); x = x + h
        h  = self.drop(F.relu(self.bn2(self.conv2(x, ei)))); x = x + h
        h  = self.drop(F.relu(self.bn3(self.conv3(x, ei)))); x = x + h
        sf, attn = self.attn(x, batch)
        return self.head(sf), attn


class WideGNN(nn.Module):
    """2-layer with wider hidden dims."""
    def __init__(self, in_dim=2048, hidden=512, n_classes=2, dropout=0.3):
        super().__init__()
        self.proj  = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(dropout)
        )
        self.conv1 = GCNConv(hidden, hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.drop  = nn.Dropout(dropout)
        self.attn  = ABMILAttention(hidden, hidden // 2)
        self.head  = nn.Sequential(
            nn.Linear(hidden, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, n_classes)
        )

    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        x  = self.proj(x)
        h  = self.drop(F.relu(self.bn1(self.conv1(x, ei)))); x = x + h
        h  = F.relu(self.bn2(self.conv2(x, ei)));            x = x + h
        sf, attn = self.attn(x, batch)
        return self.head(sf), attn

print('Model variants defined.')

## 3. Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    """One training epoch."""
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        logits, _ = model(batch)
        labels = batch.y.squeeze()
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)


@torch.no_grad()
def evaluate(model, loader, device):
    """Evaluate on a loader."""
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        batch = batch.to(device)
        logits, _ = model(batch)
        all_logits.append(logits.cpu())
        all_labels.append(batch.y.cpu())
    
    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    probs = torch.softmax(all_logits, dim=1)[:, 1].numpy()
    preds = all_logits.argmax(dim=1).numpy()
    labels_np = all_labels.numpy()
    
    auc = roc_auc_score(labels_np, probs) if len(np.unique(labels_np)) > 1 else 0.5
    f1 = f1_score(labels_np, preds, zero_division=0)
    
    return {'auc': auc, 'f1': f1, 'probs': probs, 'labels': labels_np}


def train_full(model, train_loader, val_loader, device, epochs=60, lr=1e-4, 
               scheduler_type='cosine', save_path=None, verbose=True):
    """Full training loop."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    if scheduler_type == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    else:
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    
    best_auc, best_epoch = 0, 0
    hist = {'train_loss': [], 'val_auc': [], 'val_f1': []}
    
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_metrics = evaluate(model, val_loader, device)
        scheduler.step()
        
        hist['train_loss'].append(train_loss)
        hist['val_auc'].append(val_metrics['auc'])
        hist['val_f1'].append(val_metrics['f1'])
        
        if verbose and epoch % 10 == 0:
            print(f'  Epoch {epoch:2d} | loss={train_loss:.4f} | AUC={val_metrics["auc"]:.4f} | F1={val_metrics["f1"]:.4f}')
        
        if val_metrics['auc'] > best_auc:
            best_auc = val_metrics['auc']
            best_epoch = epoch
            if save_path:
                torch.save({'epoch': epoch, 'model_state': model.state_dict()}, save_path)
    
    return best_auc, best_epoch, hist

print('Training functions defined.')

## 4. Systematic Hyperparameter Search

In [ ]:
# Experiment configurations
experiments = [
    # (name, model_class, lr, batch_size, epochs, hidden, dropout)
    ('baseline_2layer_256', ClassicalGNN, 1e-4, 4, 60, 256, 0.3),
    ('baseline_2layer_256_long', ClassicalGNN, 1e-4, 4, 80, 256, 0.3),
    ('lower_lr_1e5', ClassicalGNN, 1e-5, 4, 80, 256, 0.3),
    ('higher_lr_5e4', ClassicalGNN, 5e-4, 4, 60, 256, 0.3),
    ('smaller_bs', ClassicalGNN, 1e-4, 2, 80, 256, 0.3),
    ('wider_512', ClassicalGNN, 1e-4, 4, 60, 512, 0.3),
    ('more_dropout_0.5', ClassicalGNN, 1e-4, 4, 60, 256, 0.5),
    ('less_dropout_0.2', ClassicalGNN, 1e-4, 4, 60, 256, 0.2),
    ('deeper_3layer', DeeperGNN, 1e-4, 4, 60, 256, 0.4),
    ('deeper_3layer_long', DeeperGNN, 1e-4, 4, 80, 256, 0.4),
    ('wider_512_deeper', DeeperGNN, 1e-4, 4, 60, 512, 0.4),
]

results = {}

for name, model_class, lr, bs, epochs, hidden, dropout in experiments:
    print(f'\n{'='*60}')
    print(f'Exp: {name}')
    print(f'  Config: Model={model_class.__name__}, LR={lr:.0e}, BS={bs}, Epochs={epochs}, Hidden={hidden}, Dropout={dropout}')
    print(f'{'='*60}')
    
    # Create loaders
    tr_ldr = PyGDataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=0)
    va_ldr = PyGDataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=0)
    te_ldr = PyGDataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=0)
    
    # Create and train model
    model = model_class(in_dim=2048, hidden=hidden, dropout=dropout).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Params: {n_params:,}')
    
    best_val_auc, best_epoch, hist = train_full(
        model, tr_ldr, va_ldr, DEVICE, epochs=epochs, lr=lr, verbose=False
    )
    
    # Evaluate on test
    ckpt_path = CKPT_DIR / f'exp_{name}.pth'
    torch.save({'epoch': best_epoch, 'model_state': model.state_dict()}, ckpt_path)
    
    # Load best model
    model_best = model_class(in_dim=2048, hidden=hidden, dropout=dropout).to(DEVICE)
    best_model_state = torch.load(ckpt_path)['model_state']
    model_best.load_state_dict(best_model_state)
    
    test_metrics = evaluate(model_best, te_ldr, DEVICE)
    
    results[name] = {
        'model': model_class.__name__,
        'lr': lr,
        'batch_size': bs,
        'epochs': epochs,
        'hidden': hidden,
        'dropout': dropout,
        'best_val_auc': best_val_auc,
        'best_epoch': best_epoch,
        'test_auc': test_metrics['auc'],
        'test_f1': test_metrics['f1'],
    }
    
    print(f'  Best Val AUC: {best_val_auc:.4f} @ epoch {best_epoch}')
    print(f'  Test AUC: {test_metrics["auc"]:.4f} | F1: {test_metrics["f1"]:.4f}')

print(f'\n{"="*60}')
print('EXPERIMENT SUMMARY')
print(f'{"="*60}')

## 5. Results Summary

In [ ]:
# Sort by test AUC
sorted_results = sorted(results.items(), key=lambda x: x[1]['test_auc'], reverse=True)

print('\nRanked by Test AUC:')
print(f'{"Rank":<5} {"Name":<25} {"Test AUC":<12} {"Test F1":<12} {"Val AUC":<12}')
print('-' * 70)

for i, (name, metrics) in enumerate(sorted_results, 1):
    print(f'{i:<5} {name:<25} {metrics["test_auc"]:<12.4f} {metrics["test_f1"]:<12.4f} {metrics["best_val_auc"]:<12.4f}')

# Save results
with open(OUT_DIR / 'improvement_experiments.json', 'w') as f:
    json.dump(sorted_results, f, indent=2)

best_name, best_metrics = sorted_results[0]
print(f'\n{"="*60}')
print(f'BEST MODEL: {best_name}')
print(f'  Test AUC: {best_metrics["test_auc"]:.4f} (baseline: 0.6990)')
print(f'  Improvement: +{(best_metrics["test_auc"] - 0.6990) * 100:.2f}%')
print(f'{"="*60}')

## 6. Save Best Model

In [ ]:
# Load best model and save as new baseline
best_ckpt = torch.load(CKPT_DIR / f'exp_{best_name}.pth', weights_only=False)
best_config = results[best_name]

# Recreate model with best config
best_model_class = ClassicalGNN if best_config['model'] == 'ClassicalGNN' else \
                   DeeperGNN if best_config['model'] == 'DeeperGNN' else \
                   WideGNN

best_model = best_model_class(
    in_dim=2048, 
    hidden=best_config['hidden'], 
    dropout=best_config['dropout']
).to(DEVICE)

best_model.load_state_dict(best_ckpt['model_state'])

# Save improved baseline
improved_baseline_path = CKPT_DIR / 'classical_camelyon16_improved.pth'
torch.save({
    'model_state': best_model.state_dict(),
    'config': best_config,
    'model_class': best_model_class.__name__,
}, improved_baseline_path)

# Update baseline results
baseline_results = {
    'dataset': 'CAMELYON16 (stratified, 221 labeled slides)',
    'model': 'Classical GNN + ABMIL (Improved)',
    'n_train': len(train_ds),
    'n_val': len(val_ds),
    'n_test': len(test_ds),
    'previous_test_auc': 0.6990,
    'improved_test_auc': best_metrics['test_auc'],
    'improved_test_f1': best_metrics['test_f1'],
    'improvement': (best_metrics['test_auc'] - 0.6990) * 100,
    'best_experiment': best_name,
    'config': best_config,
}

with open(OUT_DIR / 'baseline_results_improved.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print(f'✓ Saved improved model: {improved_baseline_path}')
print(f'✓ Saved results: {OUT_DIR}/baseline_results_improved.json')

## 7. Visualization

In [ ]:
# Plot results
test_aucs = [results[name]['test_auc'] for name, _ in sorted_results]
names = [name for name, _ in sorted_results]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot: Test AUC comparison
colors = ['#2ecc71' if auc >= best_metrics['test_auc'] else '#3498db' for auc in test_aucs]
axes[0].barh(range(len(names)), test_aucs, color=colors, alpha=0.7)
axes[0].axvline(0.6990, color='red', linestyle='--', label='Baseline (0.699)', linewidth=2)
axes[0].set_yticks(range(len(names)))
axes[0].set_yticklabels(names, fontsize=9)
axes[0].set_xlabel('Test AUC')
axes[0].set_title('Model Comparison - Test AUC')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

# Box plot: Config parameters vs AUC
configs_key = [f"{r['hidden']}h_{r['dropout']:.1f}d" for _, r in sorted_results]
axes[1].scatter(range(len(names)), test_aucs, s=100, alpha=0.6)
axes[1].set_xticks(range(len(names)))
axes[1].set_xticklabels(configs_key, rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('Test AUC')
axes[1].set_title('AUC by Hidden/Dropout Config')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'week3_improvement_comparison.png', dpi=120, bbox_inches='tight')
print('✓ Saved comparison plot')
plt.show()